# FastHMR deployment notebook for VS Code

This notebook assumes that you have already created and selected the `fasthmr` Conda environment as the Jupyter kernel.

Recommended OS: Ubuntu Linux or WSL2 Ubuntu with an NVIDIA GPU. Native Windows is not recommended because PyTorch3D compilation is fragile there.

Before opening this notebook, run the terminal setup commands shown in the accompanying answer.

In [1]:
# CELL 1: Force and verify CUDA/GPU execution
# IMPORTANT: restart the notebook kernel before running this cell.
# CUDA_VISIBLE_DEVICES must be set before importing torch.

import os
import sys
import platform
import subprocess
from pathlib import Path

GPU_ID = "0"  # Change this to "1", "2", etc. only if you have multiple GPUs.

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["FORCE_CUDA"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

import torch

assert torch.cuda.is_available(), (
    "CUDA is NOT available in this notebook kernel. "
    "Check that you selected the fasthmr Conda kernel, installed pytorch-cuda, "
    "and that nvidia-smi works in this same terminal/WSL session."
)

DEVICE = torch.device("cuda:0")

torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("PyTorch:", torch.__version__)
print("Torch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count visible to PyTorch:", torch.cuda.device_count())
print("Selected CUDA device index:", torch.cuda.current_device())
print("Selected GPU:", torch.cuda.get_device_name(0))
print("GPU capability:", torch.cuda.get_device_capability(0))

x = torch.randn(1024, 1024, device=DEVICE)
y = x @ x.T
torch.cuda.synchronize()
print("CUDA tensor test OK:", y.is_cuda, y.shape)

try:
    print(subprocess.check_output(["nvidia-smi"], text=True)[:2000])
except Exception as e:
    raise RuntimeError(f"nvidia-smi failed inside this environment: {e}")

Python executable: /home/cris-sx/miniconda3/envs/fasthmr/bin/python
Python version: 3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35
CUDA_VISIBLE_DEVICES: 0
PyTorch: 2.4.1+cu121
Torch CUDA runtime: 12.1
CUDA available: True
CUDA device count visible to PyTorch: 1
Selected CUDA device index: 0
Selected GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU capability: (8, 9)
CUDA tensor test OK: True torch.Size([1024, 1024])
Thu Apr 30 10:18:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.102.01             Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                 

In [2]:
# CELL 2: Define project paths

from pathlib import Path
import os

PROJECT_ROOT = Path.home() / "UA" / "master-ia" / "tava" / "fasthmr-method" / "fasthmr-project" / "FastHMR"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

PROJECT_ROOT = /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR
OUTPUT_ROOT  = /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR/outputs
Exists: True


In [3]:
# CELL 3: Clone or update the official FastHMR repository
import subprocess
from pathlib import Path

repo_url = "https://github.com/TaatiTeam/FastHMR.git"
PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

print("Repository ready at:", PROJECT_ROOT)

Already up to date.
Repository ready at: /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR


In [ ]:
"""
# CELL 4: Install FastHMR Python requirements without replacing GPU PyTorch
# The official requirements.txt includes unpinned torch and torchvision.
# Installing it directly can overwrite the CUDA-enabled PyTorch build.
# This cell filters out torch/torchvision/torchaudio/pytorch3d/numpy/scipy and keeps the GPU stack fixed.

import sys
import subprocess
from pathlib import Path

req = PROJECT_ROOT / "requirements.txt"
assert req.exists(), f"requirements.txt not found at {req}"

# Keep NumPy/SciPy versions compatible with older SMPL/chumpy-style dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==1.23.5", "scipy==1.10.1"], check=True)

raw = req.read_text(encoding="utf-8").replace("\n", " ").strip()
tokens = raw.split()

items = []
i = 0
while i < len(tokens):
    # Preserve PEP 508 entries such as: chumpy @ git+https://...
    if i + 2 < len(tokens) and tokens[i + 1] == "@":
        items.append(tokens[i] + " @ " + tokens[i + 2])
        i += 3
    else:
        items.append(tokens[i])
        i += 1

blocked = {"torch", "torchvision", "torchaudio", "pytorch3d", "numpy", "scipy"}
filtered = []

for item in items:
    name = item.split(" @ ")[0]
    name = name.split("[")[0]
    for sep in ["==", ">=", "<=", "~=", ">", "<"]:
        name = name.split(sep)[0]
    normalized = name.strip().lower().replace("_", "-")
    if normalized not in blocked:
        filtered.append(item)

filtered_req = OUTPUT_ROOT / "requirements_no_torch_no_numpy_no_scipy.txt"
filtered_req.write_text("\n".join(filtered) + "\n", encoding="utf-8")

print("Filtered requirements written to:", filtered_req)
print(filtered_req.read_text())

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(filtered_req)], check=True)

# Verify in a fresh Python process that CUDA-enabled torch survived the pip installs.
check_code = """
import torch
print('torch:', torch.__version__)
print('torch cuda runtime:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA disappeared after installing requirements.'
"""
subprocess.run([sys.executable, "-c", check_code], check=True)

# Install extra notebook/runtime helpers.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "ipykernel", "jupyter"], check=True)

print("FastHMR Python requirements installed without replacing GPU PyTorch.")
"""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 5.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 4.2 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scipy]32m1/2 [scipy]
  Cloning https://github.com/mattloper/chumpy to /tmp/pip-install-o2zrkvke/chumpy_6820d70db90c478a9c004ae8741173bd


  Running command git clone --filter=blob:none --quiet https://github.com/mattloper/chumpy /tmp/pip-install-o2zrkvke/chumpy_6820d70db90c478a9c004ae8741173bd


  Resolved https://github.com/mattloper/chumpy to commit 580566eafc9ac68b2614b64d6f7aaa84eebb70da
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "<string>", line 9, in <module>
      ModuleNotFoundError: No module named 'pip'
      
      During handling of the above exception, another exception occurred:
      
      Traceback (most recent call last):
        File "/home/cris-sx/miniconda3/envs/fasthmr/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/home/cris-sx/miniconda3/envs/fasthmr/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
        File "/home/cris-sx/miniconda3/envs/fasthmr/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
        

CalledProcessError: Command '['/home/cris-sx/miniconda3/envs/fasthmr/bin/python', '-m', 'pip', 'install', '-r', '/home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR/requirements.txt']' returned non-zero exit status 1.

In [ ]:
"""
# CELL 5: Verify PyTorch3D GPU installation
# PyTorch3D was already installed from the terminal.
# This cell only verifies that the current notebook kernel can use PyTorch3D with CUDA.

import sys
import torch
import pytorch3d
from pytorch3d.ops import knn_points

print("Python executable:", sys.executable)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

assert torch.cuda.is_available(), "CUDA is not available in this notebook kernel."

x = torch.randn(1, 100, 3, device="cuda")
y = torch.randn(1, 200, 3, device="cuda")
out = knn_points(x, y, K=1)

print("PyTorch3D import OK")
print("PyTorch3D CUDA op OK:", out.dists.is_cuda)
assert out.dists.is_cuda
"""

In [4]:
# CELL 6: Download FastHMR pretrained checkpoints from Hugging Face
import subprocess
import sys
from pathlib import Path

checkpoints_dir = PROJECT_ROOT / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
subprocess.run([
    "hf", "download", "SoroushMehraban/FastHMR",
    "--local-dir", str(checkpoints_dir)
], check=True)

print("Downloaded checkpoints to:", checkpoints_dir)
for p in sorted(checkpoints_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(checkpoints_dir))

  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached typer-0.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)


  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)


  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 1.1 MB/s  0:00:0036m-:--:--
Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.2 MB)
Using cached typer-0.25.0-py3-none-any.whl (55 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [huggingface_hub] [huggingface_hub]


Fetching 7 files:  29%|██▊       | 2/7 [06:35<16:29, 197.85s/it]


KeyboardInterrupt: 

In [3]:
# CELL 7: Download CameraHMR/HMR2 required assets with progress bars and validation
# This cell downloads CameraHMR/HMR2 assets using authenticated requests.
# You must use your CameraHMR account credentials, NOT your Hugging Face token.

import os
import sys
import time
import getpass
import subprocess
from pathlib import Path

# Install tqdm if missing
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)

import requests
from tqdm.auto import tqdm
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

feature_dir = PROJECT_ROOT / "checkpoints" / "feature_extractors"
body_dir = PROJECT_ROOT / "checkpoints" / "body_models"
feature_dir.mkdir(parents=True, exist_ok=True)
body_dir.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("feature_dir:", feature_dir)
print("body_dir:", body_dir)

username = input("CameraHMR username/email: ").strip()
password = getpass.getpass("CameraHMR password: ")

if not username or not password:
    raise ValueError("CameraHMR username/password cannot be empty.")

session = requests.Session()

# Larger chunks reduce Python overhead and usually improve speed.
CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB
MAX_RETRIES = 5
TIMEOUT = (30, 300)  # connect timeout, read timeout


def is_probably_html_or_error_file(path: Path) -> bool:
    """
    Detects the common failure case where the server returns a login/error HTML page
    instead of the real checkpoint.
    """
    if not path.exists() or path.stat().st_size == 0:
        return True

    with open(path, "rb") as f:
        head = f.read(512).lower()

    html_markers = [
        b"<!doctype html",
        b"<html",
        b"<head",
        b"<body",
        b"login",
        b"password",
        b"invalid",
        b"error",
    ]
    return any(m in head for m in html_markers)


def download_camera_hmr_file(sfile: str, dst: Path, min_size_mb: int = 1):
    """
    Downloads a protected file from CameraHMR with progress bar and validation.
    The file is first written to .part and moved to final path only after validation.
    """
    url = f"https://download.is.tue.mpg.de/download.php?domain=camerahmr&sfile={sfile}"
    tmp = dst.with_suffix(dst.suffix + ".part")

    # Skip if already valid
    if dst.exists() and dst.stat().st_size >= min_size_mb * 1024 * 1024 and not is_probably_html_or_error_file(dst):
        print(f"Already exists and looks valid: {dst} ({dst.stat().st_size / 1024**3:.2f} GB)")
        return

    # Remove invalid final file
    if dst.exists():
        print(f"Removing invalid or too-small file: {dst}")
        dst.unlink()

    # Remove old partial if it is tiny or likely HTML
    if tmp.exists() and tmp.stat().st_size < 1024 * 1024:
        tmp.unlink()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"\nDownloading {sfile}")
            print(f"Destination: {dst}")
            print(f"Attempt {attempt}/{MAX_RETRIES}")

            with session.post(
                url,
                data={"username": username, "password": password},
                stream=True,
                verify=False,
                timeout=TIMEOUT,
                headers={
                    "User-Agent": "Mozilla/5.0",
                    "Connection": "keep-alive",
                },
            ) as r:
                print("HTTP status:", r.status_code)
                print("Content-Type:", r.headers.get("Content-Type"))
                print("Content-Length:", r.headers.get("Content-Length"))

                r.raise_for_status()

                total = int(r.headers.get("Content-Length", 0))
                downloaded = 0

                with open(tmp, "wb") as f, tqdm(
                    total=total if total > 0 else None,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=sfile,
                    dynamic_ncols=True,
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            pbar.update(len(chunk))

            if tmp.stat().st_size < min_size_mb * 1024 * 1024:
                raise RuntimeError(
                    f"Downloaded file is too small: {tmp.stat().st_size / 1024**2:.2f} MB. "
                    "This is probably an HTML login/error page, not the real file."
                )

            if is_probably_html_or_error_file(tmp):
                with open(tmp, "rb") as f:
                    preview = f.read(500)
                raise RuntimeError(
                    "Downloaded file looks like HTML/login/error page, not a checkpoint.\n"
                    f"First bytes:\n{preview!r}"
                )

            tmp.rename(dst)
            print(f"OK: {dst} ({dst.stat().st_size / 1024**3:.2f} GB)")
            return

        except Exception as e:
            print(f"Download failed on attempt {attempt}/{MAX_RETRIES}: {repr(e)}")
            if attempt == MAX_RETRIES:
                raise
            sleep_s = 10 * attempt
            print(f"Retrying in {sleep_s} seconds...")
            time.sleep(sleep_s)


def download_public_file(url: str, dst: Path, min_size_kb: int = 1):
    """
    Downloads a public file with progress bar.
    """
    if dst.exists() and dst.stat().st_size >= min_size_kb * 1024:
        print(f"Already exists: {dst} ({dst.stat().size if False else dst.stat().st_size / 1024**2:.2f} MB)")
        return

    tmp = dst.with_suffix(dst.suffix + ".part")

    with session.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))

        with open(tmp, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=dst.name,
            dynamic_ncols=True,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    if tmp.stat().st_size < min_size_kb * 1024:
        raise RuntimeError(f"Downloaded file is too small: {tmp}")

    tmp.rename(dst)
    print(f"OK: {dst} ({dst.stat().st_size / 1024**2:.2f} MB)")


# CameraHMR protected assets.
# The source filename is the one used by the CameraHMR server.
# The destination filename is the one expected by FastHMR.
camera_hmr_assets = [
    {
        "sfile": "camerahmr_checkpoint_cleaned.ckpt",
        "dst": feature_dir / "cam_model_cleaned.ckpt",
        "min_size_mb": 500,
    },
    {
        "sfile": "smpl_mean_params.npz",
        "dst": body_dir / "smpl_mean_params.npz",
        "min_size_mb": 1,
    },
    {
        "sfile": "SMPL_NEUTRAL.pkl",
        "dst": body_dir / "SMPL_NEUTRAL.pkl",
        "min_size_mb": 1,
    },
]

for asset in camera_hmr_assets:
    download_camera_hmr_file(
        sfile=asset["sfile"],
        dst=asset["dst"],
        min_size_mb=asset["min_size_mb"],
    )


# Public H36M joint regressor used by FastHMR constants.py
reg_url = (
    "https://openmmlab-share.oss-cn-hangzhou.aliyuncs.com/mmhuman3d/models/"
    "J_regressor_h36m.npy?versionId=CAEQHhiBgIDE6c3V6xciIDdjYzE3MzQ4MmU4MzQyNmRiZDA5YTg2YTI5YWFkNjRi"
)
reg_dst = body_dir / "J_regressor_h36m.npy"
download_public_file(reg_url, reg_dst, min_size_kb=10)


print("\nFinal check:")
required = [
    feature_dir / "cam_model_cleaned.ckpt",
    body_dir / "smpl_mean_params.npz",
    body_dir / "SMPL_NEUTRAL.pkl",
    body_dir / "J_regressor_h36m.npy",
]

for p in required:
    if p.exists() and p.stat().st_size > 1024 and not is_probably_html_or_error_file(p):
        print(f"OK: {p.relative_to(PROJECT_ROOT)} ({p.stat().st_size / 1024**2:.2f} MB)")
    else:
        print(f"MISSING OR INVALID: {p.relative_to(PROJECT_ROOT)}")
        raise FileNotFoundError(f"Missing or invalid file: {p}")

print("CameraHMR/HMR2 assets ready.")

PROJECT_ROOT: /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR
feature_dir: /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR/checkpoints/feature_extractors
body_dir: /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR/checkpoints/body_models

Destination: /home/cris-sx/UA/master-ia/tava/fasthmr-method/fasthmr-project/FastHMR/checkpoints/feature_extractors/cam_model_cleaned.ckpt
Attempt 1/5
HTTP status: 200
Content-Type: application/octet-stream
Content-Length: 8049228223


camerahmr_checkpoint_cleaned.ckpt:   0%|          | 0.00/7.50G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# CELL 8: Verify the required FastHMR checkpoint tree
from pathlib import Path

required = [
    PROJECT_ROOT / "checkpoints" / "fasthmr_ckpts" / "camerahmr_diffdecoder.pth.tr",
    PROJECT_ROOT / "checkpoints" / "fasthmr_ckpts" / "hmr2_diffdecoder.pth.tr",
    PROJECT_ROOT / "checkpoints" / "fasthmr_ckpts" / "vae.pth.tr",
    PROJECT_ROOT / "checkpoints" / "feature_extractors" / "camerahmr.pth.tr",
    PROJECT_ROOT / "checkpoints" / "feature_extractors" / "hmr2.pth.tr",
    PROJECT_ROOT / "checkpoints" / "feature_extractors" / "cam_model_cleaned.ckpt",
    PROJECT_ROOT / "checkpoints" / "body_models" / "SMPL_NEUTRAL.pkl",
    PROJECT_ROOT / "checkpoints" / "body_models" / "smpl_mean_params.npz",
    PROJECT_ROOT / "checkpoints" / "body_models" / "J_regressor_h36m.npy",
]

missing = []
for p in required:
    ok = p.exists() and p.stat().st_size > 1024
    print(("OK       " if ok else "MISSING  "), p)
    if not ok:
        missing.append(p)

if missing:
    raise FileNotFoundError("Some required files are missing. Re-run checkpoint/dependency download cells.")
else:
    print("All required files are present.")

In [ ]:
# CELL 9: Import smoke test for FastHMR core modules on GPU
import sys
import os
from pathlib import Path

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pytorch3d
from model.fasthmr import FastHMR
from model.smpl.SMPL import SMPL_layer
from data.preprocess.camerahmr.camerahmr_model import CameraHMR
from ultralytics import YOLO

assert torch.cuda.is_available(), "FastHMR will fall back to CPU unless CUDA is available."
DEVICE = torch.device("cuda:0")

probe = torch.randn(8, 8, device=DEVICE)
assert probe.is_cuda

print("FastHMR imports OK.")
print("CUDA available:", torch.cuda.is_available())
print("Active GPU:", torch.cuda.get_device_name(0))
print("Allocated GPU memory MB:", round(torch.cuda.memory_allocated(0) / 1024**2, 2))
print("Reserved GPU memory MB:", round(torch.cuda.memory_reserved(0) / 1024**2, 2))

In [ ]:
# CELL 10: Locate or define a demo video
from pathlib import Path

default_video = PROJECT_ROOT / "examples" / "demo_video.mp4"

# Replace this with your own file if needed:
VIDEO_PATH = default_video

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Demo video not found at {VIDEO_PATH}. "
        "Place a .mp4 video there or set VIDEO_PATH to your own video."
    )

print("Using video:", VIDEO_PATH)

In [ ]:
# CELL 11: Run the FastHMR demo with the CameraHMR backbone on GPU
# FastHMR demo.py internally uses: torch.device('cuda' if torch.cuda.is_available() else 'cpu').
# This cell passes CUDA_VISIBLE_DEVICES and FORCE_CUDA to the subprocess and fails if CUDA is unavailable.

import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA is not available. The demo would run on CPU, so execution is stopped."

os.chdir(PROJECT_ROOT)

video_name = VIDEO_PATH.stem
demo_out = OUTPUT_ROOT / f"demo_{video_name}_camerahmr_gpu"
demo_out.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["FORCE_CUDA"] = "1"
env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

cmd = [
    sys.executable, "demo.py",
    "--video", str(VIDEO_PATH),
    "--output_pth", str(demo_out),
    "--backbone-name", "camerahmr",
    "--batch-size", "16",
    "--clip-length", "243",
    "--token-merging-ratio", "40",
    "--visualize",
]

print("Running command on GPU:")
print(" ".join(cmd))
print("GPU:", torch.cuda.get_device_name(0))

result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, env=env)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("FastHMR GPU demo failed. Check the terminal output above.")
else:
    print("GPU demo completed:", demo_out)

In [ ]:
# CELL 12: Inspect demo outputs
from pathlib import Path
import pickle
import joblib

video_name = VIDEO_PATH.stem
demo_out = OUTPUT_ROOT / f"demo_{video_name}_camerahmr"

mesh_pkl = demo_out / f"mesh_results_{video_name}.pkl"
tracking_pth = demo_out / f"tracking_results_{video_name}.pth"
unified_mp4 = demo_out / f"{video_name}_unified.mp4"

print("mesh_pkl:    ", mesh_pkl, mesh_pkl.exists())
print("tracking_pth:", tracking_pth, tracking_pth.exists())
print("unified_mp4: ", unified_mp4, unified_mp4.exists())

with open(mesh_pkl, "rb") as f:
    mesh_results = pickle.load(f)

print("Detected person IDs:", list(mesh_results.keys()))
for pid, data in mesh_results.items():
    print("Person", pid)
    print("  vertices:", tuple(data["vertices"].shape))
    print("  joints:  ", tuple(data["joints"].shape))

In [ ]:
# CELL 13: Optional GPU run with HMR2 backbone
# Use this if you want to compare the two available FastHMR feature extractors.

import os
import sys
import subprocess
from pathlib import Path
import torch

RUN_HMR2 = False

if RUN_HMR2:
    assert torch.cuda.is_available(), "CUDA is not available. HMR2 run stopped."

    os.chdir(PROJECT_ROOT)
    video_name = VIDEO_PATH.stem
    demo_out_hmr2 = OUTPUT_ROOT / f"demo_{video_name}_hmr2_gpu"
    demo_out_hmr2.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["FORCE_CUDA"] = "1"
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    cmd = [
        sys.executable, "demo.py",
        "--video", str(VIDEO_PATH),
        "--output_pth", str(demo_out_hmr2),
        "--backbone-name", "hmr2",
        "--batch-size", "16",
        "--clip-length", "243",
        "--token-merging-ratio", "40",
        "--visualize",
    ]

    print("Running command on GPU:")
    print(" ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, env=env)
    print("Return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("FastHMR HMR2 GPU demo failed.")
    else:
        print("HMR2 GPU demo completed:", demo_out_hmr2)
else:
    print("RUN_HMR2 is False. Set it to True if you want to run this cell.")

In [ ]:
# CELL 14: GPU-compatible standard metric functions for later dataset evaluation
# These functions use torch tensors on CUDA when CUDA is available.
# Inputs may be NumPy arrays or torch tensors.

import numpy as np
import torch

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def _to_tensor(x, device=DEVICE, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.detach().to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)

def mpjpe(pred_joints, gt_joints, mask=None):
    pred = _to_tensor(pred_joints)
    gt = _to_tensor(gt_joints)
    err = torch.linalg.norm(pred - gt, dim=-1)
    if mask is not None:
        mask = _to_tensor(mask, dtype=torch.bool)
        err = err[mask]
    return float(err.mean().detach().cpu())

def pve(pred_vertices, gt_vertices, mask=None):
    pred = _to_tensor(pred_vertices)
    gt = _to_tensor(gt_vertices)
    err = torch.linalg.norm(pred - gt, dim=-1)
    if mask is not None:
        mask = _to_tensor(mask, dtype=torch.bool)
        err = err[mask]
    return float(err.mean().detach().cpu())

def pck(pred_joints, gt_joints, threshold=0.05, mask=None):
    pred = _to_tensor(pred_joints)
    gt = _to_tensor(gt_joints)
    correct = torch.linalg.norm(pred - gt, dim=-1) < threshold
    if mask is not None:
        mask = _to_tensor(mask, dtype=torch.bool)
        correct = correct[mask]
    return float(correct.float().mean().detach().cpu())

def _batch_similarity_transform(pred, gt):
    X = _to_tensor(pred)
    Y = _to_tensor(gt)

    if X.ndim == 2:
        X = X.unsqueeze(0)
    if Y.ndim == 2:
        Y = Y.unsqueeze(0)

    B = X.shape[0]
    device = X.device

    muX = X.mean(dim=1, keepdim=True)
    muY = Y.mean(dim=1, keepdim=True)
    X0 = X - muX
    Y0 = Y - muY

    normX = torch.sqrt((X0 ** 2).sum(dim=(1, 2), keepdim=True)).clamp_min(1e-8)
    normY = torch.sqrt((Y0 ** 2).sum(dim=(1, 2), keepdim=True)).clamp_min(1e-8)

    Xn = X0 / normX
    Yn = Y0 / normY

    H = Xn.transpose(1, 2) @ Yn
    U, S, Vh = torch.linalg.svd(H)

    R = U @ Vh
    det = torch.linalg.det(R)

    Z = torch.eye(3, device=device).unsqueeze(0).repeat(B, 1, 1)
    Z[:, -1, -1] = torch.sign(det)

    R = U @ Z @ Vh
    trace = (S * torch.diagonal(Z, dim1=-2, dim2=-1)).sum(dim=1)

    scale = trace.view(B, 1, 1) * normY / normX
    t = muY - scale * (muX @ R)

    return scale * (X @ R) + t

def pa_mpjpe(pred_joints, gt_joints):
    pred = _to_tensor(pred_joints)
    gt = _to_tensor(gt_joints)

    pred = pred.reshape(-1, pred.shape[-2], 3)
    gt = gt.reshape(-1, gt.shape[-2], 3)

    aligned = _batch_similarity_transform(pred, gt)
    err = torch.linalg.norm(aligned - gt, dim=-1)
    return float(err.mean().detach().cpu())

def acceleration_error(pred_joints, gt_joints):
    pred = _to_tensor(pred_joints)
    gt = _to_tensor(gt_joints)

    pred_acc = pred[:-2] - 2 * pred[1:-1] + pred[2:]
    gt_acc = gt[:-2] - 2 * gt[1:-1] + gt[2:]

    err = torch.linalg.norm(pred_acc - gt_acc, dim=-1)
    return float(err.mean().detach().cpu())

print("Metric functions ready on device:", DEVICE)
print("Functions: mpjpe, pa_mpjpe, pck, pve/mpvpe, acceleration_error.")

In [ ]:
# CELL 15: Example metric usage with FastHMR outputs and dataset ground truth placeholders
# Replace gt_joints and gt_vertices with arrays loaded from 3DPW/EMDB/Human3.6M preprocessing.
import pickle
import numpy as np

with open(mesh_pkl, "rb") as f:
    mesh_results = pickle.load(f)

person_id = list(mesh_results.keys())[0]
pred_vertices = _to_numpy(mesh_results[person_id]["vertices"])  # shape: T x V x 3
pred_joints = _to_numpy(mesh_results[person_id]["joints"])      # shape: T x J x 3

print("Pred vertices:", pred_vertices.shape)
print("Pred joints:", pred_joints.shape)

# Example placeholders:
# gt_vertices = np.load("path/to/gt_vertices.npy")  # same shape/unit as pred_vertices
# gt_joints = np.load("path/to/gt_joints.npy")      # same shape/unit as pred_joints
#
# print("MPJPE:", mpjpe(pred_joints, gt_joints))
# print("PA-MPJPE:", pa_mpjpe(pred_joints, gt_joints))
# print("PVE/MPVPE:", pve(pred_vertices, gt_vertices))
# print("PCK@50mm:", pck(pred_joints, gt_joints, threshold=50.0))
# print("Acceleration error:", acceleration_error(pred_joints, gt_joints))

print("Load ground truth arrays from your dataset preprocessing to compute numerical metrics.")

In [ ]:
# CELL 16: Benchmark runtime and FPS for the GPU demo command
# This measures end-to-end pipeline time, including tracking, preprocessing, diffusion, mesh extraction, and optional visualization.

import time
import cv2
import subprocess
import sys
import os
import torch
from pathlib import Path

assert torch.cuda.is_available(), "CUDA is not available. Benchmark stopped."

BENCHMARK_VISUALIZE = False

video_name = VIDEO_PATH.stem
bench_out = OUTPUT_ROOT / f"benchmark_{video_name}_camerahmr_gpu"
bench_out.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["FORCE_CUDA"] = "1"
env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

cmd = [
    sys.executable, "demo.py",
    "--video", str(VIDEO_PATH),
    "--output_pth", str(bench_out),
    "--backbone-name", "camerahmr",
    "--batch-size", "16",
    "--clip-length", "243",
    "--token-merging-ratio", "40",
]
if BENCHMARK_VISUALIZE:
    cmd.append("--visualize")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

start = time.perf_counter()
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, env=env)
elapsed = time.perf_counter() - start

if result.returncode != 0:
    raise RuntimeError("GPU benchmark demo failed.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Frames: {num_frames}")
print(f"Elapsed seconds: {elapsed:.2f}")
print(f"End-to-end FPS: {num_frames / elapsed:.3f}")

In [ ]:
# CELL 17: Save a GPU reproducibility report for your final project
import json
import subprocess
import sys
import platform
from pathlib import Path
import torch
import os

report = {
    "method": "FastHMR",
    "repository": "https://github.com/TaatiTeam/FastHMR",
    "project_root": str(PROJECT_ROOT),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu_count_visible_to_torch": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "selected_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "selected_gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "pytorch_cuda_alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
}

try:
    commit = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True).strip()
    report["git_commit"] = commit
except Exception:
    report["git_commit"] = None

if torch.cuda.is_available():
    report["gpu_memory_allocated_mb"] = round(torch.cuda.memory_allocated(0) / 1024**2, 2)
    report["gpu_memory_reserved_mb"] = round(torch.cuda.memory_reserved(0) / 1024**2, 2)

report_path = OUTPUT_ROOT / "fasthmr_gpu_reproducibility_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved:", report_path)
print(json.dumps(report, indent=2))